In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as opt
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import wandb

wb_key='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'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

wandb.login(wb_key)


PATH = './cifar_net.pth'

batch_size=64

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(images.shape)  # torch.Size([64, 1, 28, 28])
print(labels.shape)  # torch.Size([64])


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/goose/.netrc


torch.Size([64, 1, 28, 28])
torch.Size([64])


In [3]:
class Net(nn.Module):
    def __init__(self,use_batchnorm=True):
        super(Net,self).__init__()
        self.conv1=nn.Conv2d(1,10,3)
        self.bn1=nn.BatchNorm2d(10) if use_batchnorm else nn.Identity()
        self.conv2=nn.Conv2d(10,20,3)
        self.bn2=nn.BatchNorm2d(20) if use_batchnorm else nn.Identity()

        self.fc1=nn.Linear(500,144)
        self.fc2=nn.Linear(144,72)
        self.fc3=nn.Linear(72,10)
    
    def forward(self,input):
        
        c1=F.relu(self.conv1(input))
        c1=self.bn1(c1)        
        s2=F.max_pool2d(c1,(2,2))
        c3=F.relu(self.conv2(s2))
        c3=self.bn2(c3)
        s4=F.max_pool2d(c3,2)
        s4=torch.flatten(s4,1)
        f5=F.relu(self.fc1(s4))
        f6=F.relu(self.fc2(f5))
        output=self.fc3(f6)
        return output



In [6]:


epoch=50
lr=1e-5

config={
    'epoch':epoch,
    'lr':lr
}

net=Net().to(device)
print(net)       

criterion=nn.CrossEntropyLoss()
optmizer=opt.Adam(net.parameters(),lr=lr)

def val_acc(net,test_loader):
    correct=0
    total=0
    net.eval()
    with torch.no_grad():
        for data in test_loader:
            images,labels=data
            images, labels = images.to(device), labels.to(device)
            outputs=net(images)
            _,predicted=torch.max(outputs,1)
            total+=labels.size(0)
            correct+=(predicted == labels).sum().item()
    net.train()
    return correct/total

wandb.init(project='lenet5exp',name=f'lr={lr}_Bn=T',config=config)
for epo in range(epoch):
    net.train()
    running_loss=0.0
    for i,data in enumerate(train_loader,0):
        inputs,labels=data
        inputs, labels = inputs.to(device), labels.to(device)
        optmizer.zero_grad()
        outputs=net(inputs)
        loss=criterion(outputs,labels)
        loss.backward()
        optmizer.step()

        running_loss+=loss.item()
        if i%20 == 19:
            print(f'[{epo+1},{i+1:5d}] loss={running_loss/20:.3f}')
    acc=val_acc(net,test_loader)
    wandb.log({"accuracy": acc, "loss": running_loss/len(train_loader)})
wandb.finish()
print('finish train')
torch.save(net.state_dict(), PATH)

Net(
  (conv1): Conv2d(1, 10, kernel_size=(3, 3), stride=(1, 1))
  (bn1): BatchNorm2d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(10, 20, kernel_size=(3, 3), stride=(1, 1))
  (bn2): BatchNorm2d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=500, out_features=144, bias=True)
  (fc2): Linear(in_features=144, out_features=72, bias=True)
  (fc3): Linear(in_features=72, out_features=10, bias=True)
)


[1,   20] loss=2.322
[1,   40] loss=4.634
[1,   60] loss=6.930
[1,   80] loss=9.199
[1,  100] loss=11.461
[1,  120] loss=13.695
[1,  140] loss=15.921
[1,  160] loss=18.126
[1,  180] loss=20.309
[1,  200] loss=22.486
[1,  220] loss=24.639
[1,  240] loss=26.771
[1,  260] loss=28.888
[1,  280] loss=30.979
[1,  300] loss=33.046
[1,  320] loss=35.116
[1,  340] loss=37.149
[1,  360] loss=39.146
[1,  380] loss=41.138
[1,  400] loss=43.089
[1,  420] loss=45.007
[1,  440] loss=46.898
[1,  460] loss=48.760
[1,  480] loss=50.583
[1,  500] loss=52.400
[1,  520] loss=54.161
[1,  540] loss=55.904
[1,  560] loss=57.616
[1,  580] loss=59.286
[1,  600] loss=60.925
[1,  620] loss=62.547
[1,  640] loss=64.129
[1,  660] loss=65.665
[1,  680] loss=67.175
[1,  700] loss=68.645
[1,  720] loss=70.099
[1,  740] loss=71.524
[1,  760] loss=72.899
[1,  780] loss=74.229
[1,  800] loss=75.556
[1,  820] loss=76.834
[1,  840] loss=78.091
[1,  860] loss=79.329
[1,  880] loss=80.526
[1,  900] loss=81.699
[1,  920] loss

accuracy,▁▅▆▇▇▇▇▇▇▇██████████████████████████████
loss,█▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,0.9881
loss,0.02558


finish train


In [ ]:
net.load_state_dict(torch.load(PATH, weights_only=True))
def val_acc():
    correct=0
    total=0
    with torch.no_grad():
        for data in test_loader:
            images,labels=data
            outputs=net(images)
            _,predicted=torch.max(outputs,1)
            total+=labels.size(0)
            correct+=(predicted == labels).sum().item()
    return correct//total
